# AlexNet Benchmark

use PY310, py311 not support torch.compile, py39 not support libnvrtc.so compatibility

In [1]:
model_name = "vit-torch"

import torch
from torch import nn
from torchvision import models
#import torch_mlir
import numpy as np
import iree
import iree.compiler
import iree.runtime

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll


In [2]:
    
def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=3):
    return ti(stmt, globals=globals(), number=n) * 1000 / n

## Experimental

In [3]:
import torch
from torch import nn
from torchvision import models
import pandas as pd

MAGIC_NUM = 7777e-5

device = torch.device("cuda:0")
model = models.vit_b_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * MAGIC_NUM for k, v in model.state_dict().items()})
model = model.to(device)

df = pd.DataFrame()

### PyTorch (Baseline)

In [4]:
for bs in range(1, 27):
    model = torch.compile(model, backend="inductor")
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = model(image)
    grad = torch.randn_like(output)

    print("measuring #", bs)
    baseline_f = timeit("model(image.to(device))", 333)
    baseline_b = timeit("torch.autograd.grad(output.to(device), [image.to(device)], grad.to(device), retain_graph=True)", 333)
    #print(baseline_f)
    print(baseline_b)


measuring # 1
17.47652770274573
measuring # 2
28.69224739079182
measuring # 3
35.04080139939581
measuring # 4
46.18258572633828
measuring # 5
53.25004500378896
measuring # 6
60.26708784963454
measuring # 7
67.5398921957603
measuring # 8
78.33083295305302
measuring # 9
85.16432656752723
measuring # 10
214.68420735876393
measuring # 11
214.22922335475593
measuring # 12
224.31575229206868
measuring # 13
227.61784565520358
measuring # 14
235.3011485575153
measuring # 15
250.67715616980652
measuring # 16
140.5009069443644
measuring # 17
261.8140939863639
measuring # 18
152.46833403043829
measuring # 19
159.15469582228957
measuring # 20
172.53682479631524
measuring # 21
175.70835640849742
measuring # 22
183.88107624855843
measuring # 23
191.18581910536543
measuring # 24
202.30194465355115
measuring # 25
213.2735593082221
measuring # 26
219.29617522580847


Process ForkProcess-6:
Process ForkProcess-28:
Process ForkProcess-32:
Process ForkProcess-23:
Process ForkProcess-3:
Process ForkProcess-27:
Process ForkProcess-2:
Process ForkProcess-26:
Process ForkProcess-13:
Process ForkProcess-11:
Process ForkProcess-21:
Process ForkProcess-31:
Process ForkProcess-5:
Process ForkProcess-16:
Process ForkProcess-22:
Process ForkProcess-15:
Process ForkProcess-12:
Process ForkProcess-17:
Process ForkProcess-30:
Process ForkProcess-8:
Process ForkProcess-4:
Process ForkProcess-19:
Process ForkProcess-10:
Process ForkProcess-18:
Process ForkProcess-29:
Process ForkProcess-25:
Process ForkProcess-9:
Process ForkProcess-1:
Process ForkProcess-7:
Process ForkProcess-20:
Process ForkProcess-24:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most rece

In [1]:
print(ragdoll_results)

NameError: name 'ragdoll_results' is not defined